# Notebook 4 — LLM-as-a-Judge con mitigación de sesgos

**Autor:** Agustín  
**Módulo:** M2  
**Modelo evaluado:** `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` fine-tuneado con LoRA sobre DisTEMIST  
**Tarea del modelo:** NER de enfermedades en texto clínico en español (token classification, salida BIO)

## Objetivo

Implementar un juez LLM (Gemini Flash) sobre un subconjunto de ejemplos «ricos» del test set,
e identificar y mitigar **al menos 2 sesgos conocidos** del juez:

1. **Sesgo de posición**: el juez favorece la respuesta que aparece primero en el prompt.
2. **Sesgo de longitud**: el juez premia respuestas más largas independientemente de su corrección.
3. **Sesgo de auto-preferencia** (documentado): el juez podría favorecer el estilo de output de su propia familia.

El entregable es la **comparación antes/después de la mitigación** con números concretos.

## Contrato de datos (acordado con el equipo)

- `format_gold_example(row)` → devuelve `esperado` como **lista** (`list[str]`), no string.
- `select_rich_examples()` filtra sobre el mismo split `test` que usa Luis para `micro_prf1_by_doc`.
  Esta asimetría se documenta explícitamente: el subset no es comparable directamente al F1 global.
- La conversión lista → string ocurre en el prompt-building con `', '.join(lista)`,
  **no** en `format_gold_example()` — para respetar el contrato de datos del equipo.


## 0. Setup e imports

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("[INFO] No estamos en Colab — ajusta BASE_DIR manualmente.")


In [ ]:
import subprocess, sys

pkgs = ["google-generativeai", "peft", "datasets", "transformers", "accelerate"]
for pkg in pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


In [ ]:
import re, json, random, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import defaultdict

import google.generativeai as genai
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Versiones de librerías:")
import transformers as _tr, datasets as _ds, peft as _peft
print(f"  transformers: {_tr.__version__}")
print(f"  datasets:     {_ds.__version__}")
print(f"  peft:         {_peft.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  numpy:        {np.__version__}")


In [ ]:
# ============================================================
#  CONFIGURACIÓN — ajustar solo estas variables
# ============================================================

BASE_DIR   = Path("/content/drive/MyDrive/TopicosIA")
SPLITS_DIR = BASE_DIR / "distemist_final"
MODEL_DIR  = BASE_DIR / "saved_models" / "clinical_bert-distemist-lora"

BASE_CHECKPOINT = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"

# Juez LLM
GEMINI_API_KEY = ""   # <-- pegar la API key aquí
JUDGE_MODEL    = "gemini-2.0-flash"  # versión fija para reproducibilidad

MIN_UNIQUE_ENTITIES = 3
N_RICH_EXAMPLES     = 30
JUDGE_TEMPERATURE   = 0.0   # determinístico
JUDGE_MAX_TOKENS    = 512

print(f"BASE_DIR:  {BASE_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"Juez:      {JUDGE_MODEL}")


In [ ]:
import os

api_key = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY", "")
if not api_key:
    raise ValueError(
        "No se encontró la API key de Gemini. "
        "Pegala en GEMINI_API_KEY o setea GOOGLE_API_KEY."
    )
genai.configure(api_key=api_key)
judge_llm = genai.GenerativeModel(JUDGE_MODEL)
print(f"Juez LLM inicializado: {JUDGE_MODEL}")


## 1. Cargar datos y modelo

### 1.1 Helpers reutilizados de M1 (mismas funciones que `clinical_BERT/03_finetuning.ipynb`)


In [ ]:
def strip_chunk_suffix(doc_id: str) -> str:
    """Elimina el sufijo _chunkN para obtener el ID del documento original."""
    return re.sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens: list, tags: list) -> set:
    """Convierte secuencia BIO (tokens, tags) a set de strings de entidades."""
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(e.lower().strip() for e in entities)


def aggregate_entities_by_original_doc(doc_ids: list, entity_sets: list) -> dict:
    """Junta los sets de entidades de todos los chunks del mismo documento."""
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


print("Helpers de M1 cargados.")


### 1.2 Cargar el eval set (split test)


In [ ]:
def load_eval_set(splits_dir: Path):
    """
    Carga el split 'test' del dataset en formato BERT (distemist_bert_format).
    Formato de cada fila: doc_id (str), tokens (list[str]), ner_tags (list[int]).
    """
    dataset_path = splits_dir / "distemist_bert_format"
    ds = load_from_disk(str(dataset_path))
    print(f"Splits disponibles: {list(ds.keys())}")
    print(f"Tamaño split 'test': {len(ds['test'])} chunks")
    print(f"Columnas: {ds['test'].column_names}")
    return ds


ds      = load_eval_set(SPLITS_DIR)
test_ds = ds["test"]

label_names = test_ds.features["ner_tags"].feature.names
id2label    = {i: l for i, l in enumerate(label_names)}
label2id    = {l: i for i, l in id2label.items()}
print("Etiquetas:", id2label)


### 1.3 `format_gold_example` — contrato de datos del equipo

> **Contrato:** `format_gold_example()` devuelve SIEMPRE una **lista** (`list[str]`), no un string.
> Si el juez necesita un string legible para el prompt, usar `', '.join(lista)` en el prompt-building.


In [ ]:
def format_gold_example(row: dict, id2label: dict) -> list:
    """
    Devuelve la lista de entidades gold de una fila del test set.

    Contrato de datos (acordado con el equipo):
        → Devuelve SIEMPRE list[str], no string.
        → Si el juez necesita un string: usar ', '.join(...) en el prompt-building,
          NO modificar esta función.
    """
    tokens   = row["tokens"]
    ner_tags = row["ner_tags"]
    if ner_tags and isinstance(ner_tags[0], int):
        tag_strings = [id2label[t] for t in ner_tags]
    else:
        tag_strings = ner_tags
    entity_set = bio_to_entity_set(tokens, tag_strings)
    return sorted(entity_set)  # orden determinístico


ejemplo      = test_ds[0]
gold_ejemplo = format_gold_example(ejemplo, id2label)
print(f"doc_id: {ejemplo['doc_id']}")
print(f"Gold (lista): {gold_ejemplo}")
assert isinstance(gold_ejemplo, list), "format_gold_example debe devolver una lista"
print("Contrato verificado: devuelve lista.")


### 1.4 Cargar modelo fine-tuneado y definir inferencia


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

tokenizer  = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)
base_model = AutoModelForTokenClassification.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model = model.to(device)
model.eval()
print(f"Modelo cargado desde: {MODEL_DIR}")


In [ ]:
def run_roberta_inference(row: dict) -> list:
    """
    Pasa una fila por el modelo RoBERTa fine-tuneado.
    Devuelve lista de entidades predichas (misma forma que format_gold_example).
    La conversión BIO -> lista permite presentar el output al juez de forma
    arquitectura-agnóstica (igual que mT5 después de mt5_output_to_entity_set).
    """
    tokens   = row["tokens"]
    encoding = tokenizer(
        tokens, is_split_into_words=True,
        return_tensors="pt", truncation=True, max_length=512,
    )
    word_ids_list = encoding.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()

    word_preds, seen = [], set()
    for idx, wid in zip(pred_ids, word_ids_list):
        if wid is None or wid in seen:
            continue
        word_preds.append(id2label[idx])
        seen.add(wid)

    entity_set = bio_to_entity_set(tokens[: len(word_preds)], word_preds)
    return sorted(entity_set)


pred_ejemplo = run_roberta_inference(test_ds[0])
print(f"Pred (lista): {pred_ejemplo}")
assert isinstance(pred_ejemplo, list)
print("Inferencia OK")


## 2. Seleccionar ejemplos ricos (rich examples)

Los ejemplos ricos son un **subconjunto** del mismo split `test` que usa Luis para `micro_prf1_by_doc`.

> **Nota explícita (para el reporte / harness de Isa):** el scorecard del juez sobre los rich examples
> NO es directamente comparable al F1 global sobre todo el test set.
> Los rich examples son un subconjunto curado — esta asimetría debe quedar explícita en el reporte.


In [ ]:
def select_rich_examples(
    test_dataset, id2label: dict, min_unique_entities: int = 3, n: int = 30,
) -> list:
    """
    Selecciona documentos con >= min_unique_entities entidades gold únicas (por doc original).
    Corre la inferencia del modelo sobre cada documento rico.

    Devuelve lista de dicts:
        doc_id : str
        n_gold : int
        gold   : list[str]  -- contrato de equipo: lista, no string
        pred   : list[str]
        chunks : list[dict] -- filas originales del dataset
    """
    doc_chunks = defaultdict(list)
    for row in test_dataset:
        doc_chunks[strip_chunk_suffix(row["doc_id"])].append(row)

    rich_docs = []
    for orig_id, chunks in doc_chunks.items():
        gold_set = set()
        for chunk in chunks:
            gold_set.update(format_gold_example(chunk, id2label))
        if len(gold_set) >= min_unique_entities:
            rich_docs.append({
                "doc_id": orig_id, "n_gold": len(gold_set),
                "gold": sorted(gold_set), "chunks": chunks,
            })

    rich_docs.sort(key=lambda d: d["n_gold"], reverse=True)
    rich_docs = rich_docs[:n]

    print(f"Corriendo inferencia sobre {len(rich_docs)} documentos ricos...")
    for doc in rich_docs:
        pred_set = set()
        for chunk in doc["chunks"]:
            pred_set.update(run_roberta_inference(chunk))
        doc["pred"] = sorted(pred_set)

    print(f"Documentos ricos seleccionados: {len(rich_docs)}")
    counts = [d["n_gold"] for d in rich_docs]
    print(f"Entidades gold — media: {np.mean(counts):.1f}, máx: {max(counts)}")
    return rich_docs


rich_examples = select_rich_examples(
    test_ds, id2label=id2label,
    min_unique_entities=MIN_UNIQUE_ENTITIES,
    n=N_RICH_EXAMPLES,
)

for doc in rich_examples[:3]:
    print(f"\ndoc_id: {doc['doc_id']} | n_gold: {doc['n_gold']}")
    print(f"  Gold: {doc['gold']}")
    print(f"  Pred: {doc['pred']}")


## 3. RUBRICA del juez

Evalúa en 4 dimensiones (1–5 cada una):

| Dimensión | Descripción |
|---|---|
| **Completitud** | ¿Capturó la mayoría de las enfermedades mencionadas? |
| **Exactitud de boundary** | ¿Los nombres coinciden con el gold (o casi)? |
| **Relevancia clínica** | ¿Las predicciones son términos clínicamente válidos? |
| **Ausencia de ruido** | ¿Evitó marcar términos que NO son enfermedades? |

Score final = promedio de las 4 dimensiones.


In [ ]:
RUBRICA_SYSTEM = (
    "Eres un experto en NLP clínico en español. "
    "Evalúa qué tan bien un sistema de NER detectó enfermedades en texto clínico, "
    "comparando sus predicciones con una lista gold. "
    "Evalúa SOLO el contenido. No sabes qué modelo generó las predicciones."
)

RUBRICA_TEMPLATE = """
# Evaluación de NER clínico — RUBRICA

## Referencia (gold standard)
Enfermedades correctas: {gold_str}

## Predicción del sistema
Enfermedades detectadas: {pred_str}

## Instrucciones
Evalúa en cada dimensión del 1 al 5:

1. **Completitud** (¿capturó la mayoría de las enfermedades gold?):
   5=todas/casi todas | 3=la mitad | 1=prácticamente nada

2. **Exactitud de boundary** (¿los nombres coinciden con el gold?):
   5=coincidencia exacta o diferencia mínima | 3=algunos coinciden, otros truncados | 1=no se parecen

3. **Relevancia clínica** (¿las predicciones son términos de enfermedades válidos?):
   5=todas válidas | 3=mezcla | 1=mayoría inválidas

4. **Ausencia de ruido** (¿evitó marcar términos que NO son enfermedades?):
   5=sin FP notables | 3=algunos FP | 1=demasiados FP

## Respuesta
Responde ÚNICAMENTE en JSON, sin texto adicional:
```json
{{
  "completitud": <1-5>,
  "exactitud_boundary": <1-5>,
  "relevancia_clinica": <1-5>,
  "ausencia_ruido": <1-5>,
  "justificacion": "<una oración breve>"
}}
```
"""


def build_judge_prompt(gold: list, pred: list, order: str = "normal") -> str:
    """
    Construye el prompt para el juez.
    order='normal'   -> gold como referencia (rol estándar)
    order='inverted' -> pred como referencia (para detectar sesgo de posición)

    Nota: la conversión lista -> string ocurre AQUÍ (prompt-building),
    NO en format_gold_example(). Contrato de equipo respetado.
    """
    gold_str = ", ".join(gold) if gold else "(ninguna)"
    pred_str = ", ".join(pred) if pred else "(ninguna)"
    if order == "inverted":
        return RUBRICA_TEMPLATE.format(gold_str=pred_str, pred_str=gold_str)
    return RUBRICA_TEMPLATE.format(gold_str=gold_str, pred_str=pred_str)


def parse_judge_response(text: str) -> dict:
    try:
        m = re.search(r"```json\s*({.*?})\s*```", text, re.DOTALL)
        return json.loads(m.group(1) if m else text.strip())
    except Exception:
        return {"completitud": None, "exactitud_boundary": None,
                "relevancia_clinica": None, "ausencia_ruido": None,
                "justificacion": f"PARSE_ERROR: {text[:200]}"}


def compute_score(d: dict) -> float:
    vals = [d.get(k) for k in ["completitud","exactitud_boundary",
                                "relevancia_clinica","ausencia_ruido"]
            if d.get(k) is not None]
    return float(np.mean(vals)) if vals else None


def call_judge(gold: list, pred: list, order: str = "normal",
               retry: int = 3, sleep_s: float = 1.0) -> dict:
    """Llama al juez LLM con reintentos y backoff exponencial."""
    prompt = build_judge_prompt(gold, pred, order=order)
    for attempt in range(retry):
        try:
            response = judge_llm.generate_content(
                [RUBRICA_SYSTEM, prompt],
                generation_config=genai.GenerationConfig(
                    temperature=JUDGE_TEMPERATURE,
                    max_output_tokens=JUDGE_MAX_TOKENS,
                ),
            )
            time.sleep(sleep_s)
            return parse_judge_response(response.text)
        except Exception as e:
            print(f"  [Intento {attempt+1}/{retry}] Error: {e}")
            time.sleep(2 ** attempt)
    return {"completitud": None, "exactitud_boundary": None,
            "relevancia_clinica": None, "ausencia_ruido": None,
            "justificacion": "JUDGE_CALL_FAILED"}


# Verificación del prompt
print(build_judge_prompt(["neumonía", "sepsis"], ["neumonía"])[:500])


## 4. Mitigación del sesgo de posición

### Qué es el sesgo de posición
El juez puede favorecer la información que aparece **primero** en el prompt independientemente de su calidad.

### Protocolo de detección
Cada ejemplo se evalúa **dos veces**:
- `order="normal"` — gold como referencia, pred como predicción (rol estándar)
- `order="inverted"` — invertimos roles: si el juez tiene sesgo, el score cambiará al poner pred en el rol de referencia

**Delta de posición** = `|score_normal − score_inverted|`

### Estrategia de mitigación
Reportar el **promedio** de `score_normal` y `score_inverted` como score final.


In [ ]:
print(f"Corriendo juez (normal + invertido) sobre {len(rich_examples)} rich examples...")
print(f"Total llamadas al juez: {len(rich_examples) * 2}\n")

position_results = []

for i, doc in enumerate(rich_examples):
    print(f"  [{i+1}/{len(rich_examples)}] {doc['doc_id']} | n_gold={doc['n_gold']} | n_pred={len(doc['pred'])}")

    result_normal    = call_judge(doc["gold"], doc["pred"], order="normal")
    score_normal     = compute_score(result_normal)

    result_inverted  = call_judge(doc["gold"], doc["pred"], order="inverted")
    score_inverted   = compute_score(result_inverted)

    delta = abs(score_normal - score_inverted) if (score_normal and score_inverted) else None
    score_mitigado = float(np.mean([s for s in [score_normal, score_inverted] if s is not None]))

    position_results.append({
        "doc_id"         : doc["doc_id"],
        "n_gold"         : doc["n_gold"],
        "n_pred"         : len(doc["pred"]),
        "gold"           : doc["gold"],
        "pred"           : doc["pred"],
        "score_normal"   : score_normal,
        "score_inverted" : score_inverted,
        "delta_posicion" : delta,
        "score_mitigado" : score_mitigado,
        "result_normal"  : result_normal,
        "result_inverted": result_inverted,
    })

print("\nListo.")


In [ ]:
df_pos = pd.DataFrame(position_results)
deltas = df_pos["delta_posicion"].dropna()

print("=" * 55)
print("ANÁLISIS — SESGO DE POSICIÓN")
print("=" * 55)
print(f"  Ejemplos evaluados          : {len(df_pos)}")
print(f"  Delta medio  |normal-inv|   : {deltas.mean():.3f}")
print(f"  Delta máximo                : {deltas.max():.3f}")
print(f"  Delta mediana               : {deltas.median():.3f}")
print(f"  % ejemplos con delta > 1    : {(deltas > 1).mean()*100:.1f}%")
print()

if deltas.mean() < 0.5:
    verdict = "ESTABLE respecto a posición (delta medio < 0.5). Mitigación es una precaución razonable."
elif deltas.mean() < 1.0:
    verdict = "SESGO MODERADO de posición (delta 0.5–1.0). Mitigación reduce el efecto."
else:
    verdict = "SESGO FUERTE de posición (delta > 1.0). Considerar cambiar el prompt."
print(f"Conclusión: {verdict}")

print("\nTop 5 con mayor delta:")
display(df_pos[["doc_id", "score_normal", "score_inverted", "delta_posicion"]]
        .sort_values("delta_posicion", ascending=False).head(5))


## 5. Mitigación del sesgo de longitud

### Qué es el sesgo de longitud
El juez puede puntuar más alto respuestas más largas, independientemente de si son correctas.

### Protocolo de detección
Pares de validación con asimetría de longitud controlada:
- **Tipo A** — Correcta corta vs. Incorrecta larga: el juez debería puntuar más alto la correcta.
- **Tipo B** — Correcta larga vs. Incorrecta corta: el juez debería puntuar más alto la correcta.


In [ ]:
LENGTH_BIAS_PAIRS = [
    # Tipo A — correcta corta vs. incorrecta larga
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["neumonía", "sepsis"],
        "pred_correcta" : ["neumonía"],
        "pred_incorrecta": ["hiperglucemia", "dislipidemia", "hipotiroidismo",
                             "artritis reumatoide", "fibromialgia",
                             "esclerosis múltiple", "enfermedad de crohn", "psoriasis"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["infarto agudo de miocardio", "insuficiencia cardiaca", "fibrilación auricular"],
        "pred_correcta" : ["infarto agudo de miocardio"],
        "pred_incorrecta": ["diabetes mellitus tipo 2", "hipertensión arterial",
                             "anemia ferropénica", "neuropatía diabética",
                             "retinopatía diabética", "enfermedad renal crónica", "osteoporosis"],
    },
    # Tipo B — correcta larga vs. incorrecta corta
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["neumonía", "sepsis", "insuficiencia respiratoria aguda"],
        "pred_correcta" : ["neumonía", "sepsis", "insuficiencia respiratoria aguda"],
        "pred_incorrecta": ["fibromialgia"],
    },
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["carcinoma ductal infiltrante", "metástasis hepática",
                  "anemia", "trombocitopenia", "neutropenia febril"],
        "pred_correcta" : ["carcinoma ductal infiltrante", "metástasis hepática",
                             "anemia", "trombocitopenia"],
        "pred_incorrecta": ["psoriasis"],
    },
    # Tipo A especial — gold vacío
    {
        "tipo": "A", "descripcion": "Correcta vacía vs. Incorrecta larga (FP puros)",
        "gold": [],
        "pred_correcta" : [],
        "pred_incorrecta": ["artritis", "hipertensión", "diabetes",
                              "anemia", "fibromialgia", "neuropatía"],
    },
]
print(f"Pares preparados: {len(LENGTH_BIAS_PAIRS)}")
for p in LENGTH_BIAS_PAIRS:
    print(f"  Tipo {p['tipo']}: n_correcta={len(p['pred_correcta'])}, n_incorrecta={len(p['pred_incorrecta'])}")


In [ ]:
print("Corriendo juez sobre pares de longitud...")
length_results = []

for i, pair in enumerate(LENGTH_BIAS_PAIRS):
    print(f"  [{i+1}/{len(LENGTH_BIAS_PAIRS)}] Tipo {pair['tipo']} — {pair['descripcion']}")
    r_cor = call_judge(pair["gold"], pair["pred_correcta"],  order="normal")
    r_inc = call_judge(pair["gold"], pair["pred_incorrecta"], order="normal")
    sc, si = compute_score(r_cor), compute_score(r_inc)
    length_results.append({
        "tipo"              : pair["tipo"],
        "descripcion"       : pair["descripcion"],
        "n_correcta"        : len(pair["pred_correcta"]),
        "n_incorrecta"      : len(pair["pred_incorrecta"]),
        "score_correcta"    : sc,
        "score_incorrecta"  : si,
        "juez_premia_calidad": (sc > si) if (sc is not None and si is not None) else None,
        "justif_correcta"   : r_cor.get("justificacion", ""),
        "justif_incorrecta" : r_inc.get("justificacion", ""),
    })

print("\nListo.")


In [ ]:
df_len  = pd.DataFrame(length_results)
n_ok    = df_len["juez_premia_calidad"].sum()
n_total = df_len["juez_premia_calidad"].notna().sum()

print("=" * 55)
print("ANÁLISIS — SESGO DE LONGITUD")
print("=" * 55)
print(f"  Pares evaluados: {n_total}")
print(f"  El juez premió calidad sobre longitud: {n_ok}/{n_total} ({n_ok/n_total*100:.0f}%)")
print()

if n_ok / n_total >= 0.8:
    print("El juez NO muestra sesgo de longitud relevante.")
elif n_ok / n_total >= 0.6:
    print("Sesgo LEVE de longitud — en algunos casos premia extensión.")
else:
    print("Sesgo FUERTE de longitud — agregar instrucción explícita al prompt.")

print()
display(df_len[["tipo", "descripcion", "n_correcta", "n_incorrecta",
               "score_correcta", "score_incorrecta", "juez_premia_calidad"]])


## 6. Sesgo de auto-preferencia (documentación)

### Qué es
Un LLM juez tiende a preferir outputs de modelos de su misma familia.

### Situación en este proyecto
- **Juez:** `gemini-2.0-flash` (familia Google)
- **Modelo evaluado:** `roberta-base-biomedical-clinical-es` (encoder, NO de Google)

El riesgo de auto-preferencia es **bajo** para RoBERTa, pero existe si el juez también evalúa mT5.

### Mitigaciones aplicadas
1. **Output anonimizado** — el prompt NO menciona el nombre del modelo.
2. **Formato estandarizado** — BIO → lista para RoBERTa, texto → lista para mT5, presentados igual.

### Limitación residual
Sin un juez de familia distinta (ej. GPT-4o) no es posible testear directamente.
Se reporta como limitación explícita.


In [ ]:
sample_prompt = build_judge_prompt(["neumonía", "sepsis"], ["neumonía"])
forbidden     = ["roberta", "bert", "mt5", "gpt", "gemini", "plantl", "biomedical", "clinical"]
found = [n for n in forbidden if n.lower() in sample_prompt.lower()]

if not found:
    print("El prompt NO menciona ningún modelo — output anonimizado.")
else:
    print(f"ATENCIÓN: el prompt menciona {found} — revisar RUBRICA_TEMPLATE.")

print()
print("Resumen de mitigaciones de auto-preferencia:")
print("  1. Output anonimizado (sin nombre de modelo en el prompt):")
print(f"     {'OK' if not found else 'PENDIENTE'}")
print("  2. Formato estandarizado (lista -> str igual para todos los modelos): OK")
print("  3. Test cross-family (juez de familia distinta): NO disponible — limitación reportada.")
print()
print("Para una evaluación más robusta se recomendaría usar al menos dos jueces")
print("de familias distintas (ej. Gemini + GPT-4o) y comparar sus scores.")


## 7. Scorecard final sobre rich examples

Usamos el **score mitigado** (promedio normal + invertido, Sección 4) como score final del juez.


In [ ]:
def compute_exact_f1_doc(gold: list, pred: list) -> dict:
    """F1 exact-match para un documento (para triangular con la métrica de Luis)."""
    g, p = set(gold), set(pred)
    tp   = len(g & p)
    fp   = len(p - g)
    fn   = len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


scorecard_rows = []
for row in position_results:
    exact = compute_exact_f1_doc(row["gold"], row["pred"])
    scorecard_rows.append({
        "doc_id"          : row["doc_id"],
        "n_gold_entities" : row["n_gold"],
        "n_pred_entities" : row["n_pred"],
        "score_juez"      : row["score_mitigado"],
        "delta_posicion"  : row["delta_posicion"],
        "f1_exacto"       : exact["f1"],
        "precision_exacta": exact["precision"],
        "recall_exacto"   : exact["recall"],
        "score_semantico" : None,   # placeholder — tarea de Pau
        "gold"            : row["gold"],
        "pred"            : row["pred"],
        "justificacion"   : row["result_normal"].get("justificacion", ""),
    })

df_sc = pd.DataFrame(scorecard_rows)

print("=" * 55)
print("SCORECARD — Rich Examples")
print("=" * 55)
print(f"  Docs evaluados           : {len(df_sc)}")
print(f"  Score juez (mitigado)    : {df_sc['score_juez'].mean():.3f} ± {df_sc['score_juez'].std():.3f}")
print(f"  F1 exacto (mismo subset) : {df_sc['f1_exacto'].mean():.3f} ± {df_sc['f1_exacto'].std():.3f}")
print()
display(df_sc[["doc_id", "n_gold_entities", "n_pred_entities",
               "score_juez", "delta_posicion",
               "f1_exacto", "precision_exacta", "recall_exacto"]])


## 8. Interpretación — ¿Qué revela el juez que el F1 exacto no capta?

| Patrón | F1 exacto | Score juez | Interpretación |
|--------|-----------|-----------|----------------|
| **A** | ≈ 0 | ≥ 3.5 | Entendió la entidad pero con **boundary distinto** — error de span, no de comprensión. |
| **B** | > 0.3 | ≤ 2.5 | Acertó la string exacta pero el output **no es clínicamente satisfactorio**. |
| **C** | ≈ 0 | ≤ 2.0 | Fallo completo. |
| **D** | ≥ 0.7 | ≥ 4.0 | Funcionamiento correcto — referencia positiva. |
| **E** | resto | resto | Caso mixto. |


In [ ]:
def classify_pattern(row):
    s, f = row["score_juez"], row["f1_exacto"]
    if s is None or f is None: return "? — datos faltantes"
    if f < 0.1 and s >= 3.5:   return "A — boundary error (comprensión OK, span malo)"
    if f > 0.3 and s <= 2.5:   return "B — string OK, calidad clínica baja"
    if f < 0.1 and s <= 2.0:   return "C — fallo completo"
    if f >= 0.7 and s >= 4.0:  return "D — referencia positiva"
    return "E — caso mixto"


df_sc["patron"] = df_sc.apply(classify_pattern, axis=1)
print("Distribución de patrones:")
print(df_sc["patron"].value_counts().to_string())

for patron_key, label in [("A", "Boundary error, comprensión OK"),
                           ("B", "String OK, calidad clínica baja")]:
    sub = df_sc[df_sc["patron"].str.startswith(patron_key)]
    print(f"\n--- Patrón {patron_key} ({len(sub)} docs) — {label} ---")
    for _, r in sub.iterrows():
        print(f"  doc: {r['doc_id'][:45]:45s} | F1={r['f1_exacto']:.2f} | Juez={r['score_juez']:.2f}")
        print(f"    Gold: {r['gold']}")
        print(f"    Pred: {r['pred']}")


In [ ]:
import json as _j

OUTPUT_DIR = BASE_DIR / "outputs" / "M2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "llm_judge_scorecard.csv"
df_sc.drop(columns=["gold", "pred"], errors="ignore").to_csv(
    csv_path, index=False, encoding="utf-8"
)
print(f"Scorecard  -> {csv_path}")

bias_summary = {
    "judge_model"        : JUDGE_MODEL,
    "base_model"         : BASE_CHECKPOINT,
    "seed"               : SEED,
    "n_rich_examples"    : len(df_sc),
    "min_unique_entities": MIN_UNIQUE_ENTITIES,
    "score_juez_mean"    : float(df_sc["score_juez"].mean()),
    "score_juez_std"     : float(df_sc["score_juez"].std()),
    "f1_exacto_mean"     : float(df_sc["f1_exacto"].mean()),
    "position_bias": {
        "delta_mean" : float(deltas.mean()),
        "delta_max"  : float(deltas.max()),
        "mitigation" : "average of normal and inverted order scores",
    },
    "length_bias": {
        "pairs_quality_wins": int(n_ok),
        "pairs_total"       : int(n_total),
        "pct_quality_wins"  : float(n_ok / n_total),
        "mitigation"        : "manual validation pairs — judge follows content, not length",
    },
    "auto_preference_bias": {
        "output_anonymized"   : True,
        "format_standardized" : True,
        "cross_family_test"   : False,
        "note": "Cannot quantify without a judge from a different model family",
    },
    "pattern_distribution": df_sc["patron"].value_counts().to_dict(),
}

json_path = OUTPUT_DIR / "llm_judge_bias_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    _j.dump(bias_summary, f, indent=2, ensure_ascii=False)
print(f"Bias summary -> {json_path}")

print("\n" + "=" * 55)
print("RESUMEN FINAL")
print("=" * 55)
print(f"  Sesgo posición  — delta medio: {deltas.mean():.3f}  | mitigación: promedio normal+invertido")
print(f"  Sesgo longitud  — calidad gana: {n_ok}/{n_total}    | mitigación: pares de validación")
print(f"  Auto-preferencia— anonimizado: OK | test cross-family: no disponible (limitación)")
print(f"  Score juez final (mitigado)  : {df_sc['score_juez'].mean():.3f}")
print(f"  F1 exacto (mismo subset)     : {df_sc['f1_exacto'].mean():.3f}")
